# cintx on a Colab T4 — CUDA runtime verification

`.planning/notes/cuda-metal-verification-gap.md` records the CUDA backend as
**compile-only**: it is built in the feature matrix and has never executed a
kernel on real hardware. Every statement about it today is a statement about the
compiler, not the device. This notebook changes that.

**What a run here settles**

1. The backend resolves and runs.
2. The **FMA-fusion probe's answer on NVIDIA**, which decides whether the
   extended Rys ceiling (`nroots` 6-12, and therefore def2-TZVP) is available.
   This is a property of the compiler's contraction behaviour and cannot be
   inferred from AMD's answer.
3. `int2e_sph` against vendored libcint 6.1.3 at the project's `1e-12`.
4. CUDA against the CPU backend, sizing the cooperative-vs-per-unit divergence.
5. The M3 device-side cart-to-sph transform: bit-identical, and reading back the
   spherical output rather than the larger Cartesian one.

**What it does not settle: anything about speed.** A T4's f64 rate is 1/32 of
its f32 rate — about 254 GFLOP/s against 8.1 TFLOP/s — and cintx's public path
is f64 by contract. A throughput number from this device would describe the T4,
not cintx. It is a correctness target, exactly as the gfx1151 development GPU is.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

Budget 25–45 minutes. The release build of the dependency tree dominates;
everything after it is minutes.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv

## 2. Get the source

cintx is not published, so bring your own copy. Either clone your remote or
upload a tarball and untar it to `/content/cintx`.

In [ ]:
# Option A — clone (replace with your remote)
# !git clone --depth 1 <your-cintx-remote> /content/cintx

# Option B — upload cintx.tar.gz via the Files pane, then:
# !mkdir -p /content/cintx && tar xzf /content/cintx.tar.gz -C /content/cintx --strip-components=1

import pathlib
assert pathlib.Path('/content/cintx/Cargo.toml').exists(), \
    'put the repo at /content/cintx first (clone or untar above)'
print('source present')

## 3. Install Rust

`rust-toolchain.toml` pins the `stable` channel, so rustup picks the right one.

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version && rustc --version

## 4. Build and verify

One script, so the whole run is reproducible outside a notebook too. It builds
with `--features cpu,cuda,extended-device-rys` and `CINTX_ORACLE_BUILD_VENDOR=1`
(which compiles vendored libcint 6.1.3 as the oracle), then runs the
verification, the device-transform check and the launch-class coverage gate.

In [ ]:
!bash /content/cintx/ci/colab_t4_verification.sh 2>&1 | tail -80

## 5. Optional — the S3 defect on a second vendor

On gfx1151, declaring a `SharedMemory` inside the batched 2e kernel corrupts its
output **even when the slab is never read**, at any size, with autotuning off —
while the identical shape round-trips correctly in an isolation probe on the same
device. Whether NVIDIA shows the same behaviour is the single most useful extra
datum a second vendor can supply, so it is offered behind an explicit opt-in.

A pass here would mean the defect is AMD-specific and S3 is viable on CUDA. A
failure would mean it is a CubeCL-level issue worth reporting upstream. Either
answer is worth having.

In [ ]:
!CINTX_TRY_SHARED_G=1 bash /content/cintx/ci/colab_t4_verification.sh 2>&1 | tail -60

## 6. What to bring back

Paste the output of section 4 (and 5 if you ran it). The lines that matter:

- `cpu  fused=… / cuda fused=…` and the `nroots ceiling` beside each — this is
  the answer that decides def2-TZVP on NVIDIA.
- The `cuda vs vendor` and `cuda vs cpu` columns.
- `readback … MiB for a … MiB output` from the device-transform test: equal
  numbers mean M3 is doing what it exists to do.